# Pakistan Weather Data Processing

This notebook processes and merges two raw datasets:
- **Temperature data** (`pakistan_city_weather_daily.csv`) — primary dataset
- **Humidity data** (`pakistan_humidity_daily.csv`) — source for `rh_avg`, `prcp`, `et0`

### Steps
1. Process humidity data — normalize timestamps, compute `rh_avg`, trim columns
2. Merge humidity into temperature (left join on `time` + `city`)
3. Resolve `prcp` conflict — drop temperature's `prcp_x`, keep humidity's `prcp_y` as `prcp`
4. StandardScale `rh_avg`, `prcp`, `et0`
5. Save final output to `data/processed/`

## 0. Imports & Path Helpers

In [ ]:
import pandas as pd
from pathlib import Path
from sklearn.preprocessing import StandardScaler

# Base directory — works in Jupyter (no __file__)
BASE_DIR = Path().resolve()

def first_existing_path(paths):
    """Return the first path that exists, or None."""
    return next((p for p in paths if p.exists()), None)

print(f"Working directory: {BASE_DIR}")

## 1. Process Humidity Data

- Normalize timestamps (UTC offset → plain date)
- Compute `rh_avg = (rh_max + rh_min) / 2`
- Trim to only columns needed for merging
- Save back to source file

In [ ]:
humidity_csv = first_existing_path([
    BASE_DIR / "pakistan_humidity_daily.csv",
    BASE_DIR.parent / "data" / "raw" / "pakistan_humidity_daily.csv",
    BASE_DIR.parent.parent / "pakistan_humidity_daily.csv",
])
if humidity_csv is None:
    raise FileNotFoundError("Could not find pakistan_humidity_daily.csv in expected locations.")

humidity_df = pd.read_csv(humidity_csv)

# Normalize timestamp — strip UTC offset, keep date only
humidity_df["time"] = pd.to_datetime(humidity_df["time"]).dt.normalize()
humidity_df["time"] = humidity_df["time"].dt.date.astype(str)

# Compute average humidity
humidity_df["rh_avg"] = (humidity_df["rh_max"] + humidity_df["rh_min"]) / 2

# Keep only columns needed for merging
humidity_df = humidity_df[["time", "city", "rh_avg", "prcp", "et0"]]

# Save back
humidity_df.to_csv(humidity_csv, index=False)

print(f"Saved → {humidity_csv}")
print(f"Shape: {humidity_df.shape}")
humidity_df.head()

## 2. Load Temperature Data (Primary)

In [ ]:
temp_csv = first_existing_path([
    BASE_DIR / "pakistan_temperature_daily.csv",
    BASE_DIR.parent / "data" / "raw" / "pakistan_city_weather_daily.csv",
])
if temp_csv is None:
    raise FileNotFoundError("Could not find temperature CSV in expected locations.")

temp_df = pd.read_csv(temp_csv)
temp_df["time"] = pd.to_datetime(temp_df["time"]).dt.date.astype(str)

print(f"Loaded: {temp_csv}")
print(f"Shape: {temp_df.shape}")
temp_df.head()

## 3. Merge + Resolve prcp Conflict

Left join keeps all temperature rows. Since both datasets have `prcp`, pandas produces `prcp_x` (temperature) and `prcp_y` (humidity). We drop `prcp_x` and rename `prcp_y` → `prcp` to use humidity's precipitation.

In [ ]:
merged_df = pd.merge(
    temp_df,
    humidity_df,
    on=["time", "city"],
    how="left"
)

# Resolve prcp conflict — keep humidity's prcp
if "prcp_y" in merged_df.columns:
    merged_df = merged_df.drop(columns=["prcp_x"]).rename(columns={"prcp_y": "prcp"})

print(f"Temp rows: {len(temp_df)}  →  Merged rows: {len(merged_df)}")
print(f"Columns: {merged_df.columns.tolist()}")
merged_df.head()

## 4. StandardScale — rh_avg, prcp, et0

In [ ]:
cols_to_scale = ["rh_avg", "prcp", "et0"]

scaler = StandardScaler()
scaled_values = scaler.fit_transform(
    merged_df[cols_to_scale].fillna(merged_df[cols_to_scale].mean())
)

merged_df[["rh_avg_scaled", "prcp_scaled", "et0_scaled"]] = scaled_values

print("Scaling complete.")
merged_df[["time", "city", "rh_avg", "rh_avg_scaled", "prcp", "prcp_scaled", "et0", "et0_scaled"]].head(10)

## 5. Save Final Output

In [ ]:
output_path = BASE_DIR.parent / "data" / "processed" / "pakistan_weather_merged_scaled.csv"
output_path.parent.mkdir(parents=True, exist_ok=True)
merged_df.to_csv(output_path, index=False)

print(f"Saved → {output_path}")
print(f"Final shape: {merged_df.shape}")
print(f"Columns: {merged_df.columns.tolist()}")